# Archivos fuente

In [0]:
# !pip install pyxlsb #

In [0]:
from os import listdir
from datetime import datetime,date
import pyspark.sql.functions as F
from pyspark.sql.types import DoubleType, IntegerType, StringType,FloatType,DateType,TimestampType, StructField, StructType
from datetime import datetime,date,timedelta
from dateutil.relativedelta import relativedelta
from pyspark.sql.window import Window
import pandas as pd
from functools import reduce
from pyspark.sql import DataFrame
import pytz
import csv
import numpy as np
import glob
import os
from openpyxl import load_workbook

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS mx_unilever_pbl;
USE SCHEMA mx_unilever_pbl;

In [0]:
def rename_columns(df,lista):
  size_list=len(lista)
  if size_list>0:
    head,tail = (lambda lista : (lista[0],lista[1:]))(lista)
    if head:
      df_clean = df.withColumnRenamed(head,head.replace("sum(","").replace(")","").replace("+","_").replace(" ","_").replace("-","_").replace(".",""))
      return rename_columns(df_clean,tail)
  else:
    return df
  
def clean_numbers(df, columns):
    for column in columns:
        df = df.withColumn(
            column,
            F.regexp_replace(F.col(column), r"\.", "")  
        ).withColumn(
            column,
            F.regexp_replace(F.col(column), r",", ".") 
        ).withColumn(
            column,
            F.col(column).cast(DoubleType())
        )
    return df


## Read data

### a) Solicitantes


In [0]:

files = sorted(glob.glob("/dbfs/mnt/unilever/PBL/OTC0023/"+"*.txt"))

df_solicitantes = pd.concat(
    (
        pd.read_csv(f, sep="\t", skiprows=3, dtype=str, encoding="cp1252").assign(File_Date=os.path.basename(f)[:7])
        for f in files
    ),
    ignore_index=True
)

df_solicitantes.columns = df_solicitantes.columns.str.strip()

# Elimina todas las columnas Unnamed
df_solicitantes = df_solicitantes.drop(
    columns=df_solicitantes.filter(regex="^Unnamed").columns
)

df_solicitantes = spark.createDataFrame(df_solicitantes)

df_solicitantes = rename_columns(
    df_solicitantes,
    df_solicitantes.columns
)

df_solicitantes = df_solicitantes.dropDuplicates([
    "Sales_Orga",
    "Division",
    "Sold_To",
    "File_Date"
])


df_solicitantes = df_solicitantes.toDF(
    "Org_Ventas",
    "Sector",
    "Holding_L5",
    "Holding_L5_Name",
    "Banner_L6",
    "Banner_L6_Name",
    "Grupo_Cliente",
    "Nombre_Grupo_Cliente",
    "Solicitante",
    "Customer_Name",
    "Pagador",
    "City",
    "State", 
    "YYYY_MM"
)

In [0]:
print(type(df_solicitantes))

<class 'pyspark.sql.classic.dataframe.DataFrame'>


In [0]:
print(df_solicitantes.show(3))

+----------+------+----------+--------------------+---------+--------------------+-------------+--------------------+-----------+--------------------+--------+---------+-----+-------+
|Org_Ventas|Sector|Holding_L5|     Holding_L5_Name|Banner_L6|      Banner_L6_Name|Grupo_Cliente|Nombre_Grupo_Cliente|Solicitante|       Customer_Name| Pagador|     City|State|YYYY_MM|
+----------+------+----------+--------------------+---------+--------------------+-------------+--------------------+-----------+--------------------+--------+---------+-----+-------+
|      MX01|    16|  10901018|OTHERS Mx_TIENDAS...| 10901037|OTHERS Mx_TIENDAS...|     30661218|    MX TIENDAS SIX B|   10110368|SILVINO CRUZ RAMIREZ|10578964|GUADALUPE|   NL|2025.07|
|      MX01|    16|  10901014|OTHERS Mx_FARMACI...| 10901033|OTHERS Mx_FARMACI...|     10901215|MX PF_TRANSPORTISTAS|   10112189|FLETES MEXICO CHI...|10112189|TULTITLAN|  MEX|2025.09|
|      MX01|    16|  10901014|OTHERS Mx_FARMACI...| 10901033|OTHERS Mx_FARMACI..

### b) Catalogos
Se obtienen los pyspark dataframes: df_menus, df_perfil_de_flota, df_grupos, df_others, df_excepcion

In [0]:
files = sorted(glob.glob("/dbfs/mnt/unilever/PBL/Catalogo/"+"*Catalogo.xlsx"))

for hoja in [
    "Menus"
    ,"Perfil_de_flota"
    ,"Grupos"
    ,"Others"
    ,"Excepcion"
    ]:

    df = pd.concat((pd.read_excel(f,sheet_name=hoja,engine="openpyxl",dtype=str).assign(yyyy_mm=os.path.basename(f)[:7]) 
        for f in files 
            if hoja != "Excepcion" or hoja 
                in load_workbook(f,read_only=True).sheetnames),ignore_index=True)

    df.columns = df.columns.str.strip()
    df = rename_columns(spark.createDataFrame(df),df.columns)

    globals()["df_"+hoja.lower()] = df

Se muestran algunas propiedades.

In [0]:
print(type(df_menus))
print(type(df_perfil_de_flota))

<class 'pyspark.sql.classic.dataframe.DataFrame'>
<class 'pyspark.sql.classic.dataframe.DataFrame'>


In [0]:
print((df_perfil_de_flota.count(), len(df_perfil_de_flota.columns)))

(40, 6)


In [0]:
df_excepcion.show(3)

+--------------+---------------------+----------+-------+
|Grupo_Clientes|Nombre_Grupo_Clientes|Comentario|yyyy_mm|
+--------------+---------------------+----------+-------+
|      10901449|             MX CORVI|       EXC|2025.09|
|      10901458|            MX DECASA|       EXC|2025.09|
|      10901463|         MX QB_RIVERA|       EXC|2025.09|
+--------------+---------------------+----------+-------+
only showing top 3 rows


## Persist data

### a) Solicitantes

In [0]:
(df_solicitantes.write.mode("overwrite").format("delta").option("header", "true")
 .option("overwriteSchema", "true").option("encoding", "UTF-8")
 .save("/mnt/adls/SupplyChain/PBL/Solicitantes_Dev"))


spark.sql("""DROP TABLE IF EXISTS solicitantes""")
spark.sql("""CREATE TABLE IF NOT EXISTS solicitantes
USING DELTA
OPTIONS
(
PATH 'dbfs:/mnt/adls/SupplyChain/PBL/Solicitantes_Dev'
)""")

DataFrame[]

### a) Grupos ?

In [0]:
for tabla in [
    "menus"
    ,"perfil_de_flota"
    ,"grupos"
    ,"others"
    ,"excepcion"
    ]:

    globals()["df_"+tabla].write.mode("overwrite").format("delta").option("header","true").option("overwriteSchema","true").option("encoding","UTF-8").save(f"/mnt/adls/SupplyChain/PBL/{tabla.title().replace('_','_')}_Dev")

    spark.sql(f"DROP TABLE IF EXISTS {tabla}")

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {tabla}
        USING DELTA
        LOCATION 'dbfs:/mnt/adls/SupplyChain/PBL/{tabla.title().replace('_','_')}_Dev'
    """)

In [0]:
%sql SELECT * FROM mx_unilever_pbl.excepcion LIMIT 10; 

Grupo_Clientes,Nombre_Grupo_Clientes,Comentario,yyyy_mm
10901449,MX CORVI,EXC,2025.09
10901458,MX DECASA,EXC,2025.09
10901463,MX QB_RIVERA,EXC,2025.09
10901476,MX PN_CENT D VIVERES,EXC,2025.09
30686343,MAYOREO DE ABARROTES JM S A DE C V,EXC,2025.09
10901510,MX PX_EL DUERO,EXC,2025.09
30094983,GUNA BPD L7,EXC,2025.11
30099977,MX ASTRID DEL TORO BPD CHART,EXC,2025.11
30099993,MX PRODUCTOS DE CONSUMO Z BPD S,EXC,2025.11
30100000,DANIELA LETICIA GUALINO BPD CHART,EXC,2025.11
